# SAC arrival_v2 — history k=8→12 monotonicity scan on single_cross_s0 (seed=42, 1M, vanilla)

**Pre-context（commit `7be0956`）**：[`docs/arrival_v2_experiment_report.md`](../docs/arrival_v2_experiment_report.md) §7.8 已闭环。`s0` + vanilla SAC + history k=4→8 单变量 ablation 在 seed=42 上 **PASS**，闭合 §7.6.4 的 80pp gap，证实 actor-side temporal info access 是瓶颈：

| §7.8 闭环              | history | final  | peak@step    | mean39 | OOB   | Gate |
|---                     |---:     |---:    |---           |---:    |---:   |---   |
| s0 + k=4 (§7.6.4 base) | 4       | 0.100  | 0.367 @ 975k | 0.221  | 0.667 | FAIL(3/5) |
| **s0 + k=8 (§7.8)**    | **8**   | **0.900** | **0.900 @ 475k** | **0.636** | **0.100** | **PASS（5/5）** |

物理量：每 control step ≈ 0.5 s（dt × eval_action_repeat）。
- k=4 → ~2s 历史窗（涡街周期 10–20 s 的 10–20%）— FAIL
- k=8 → ~4s 历史窗（20–40%）— PASS
- **k=12 → ~6s 历史窗（30–60%）** ← 本 notebook

**关键 open question**：k=4→8 不是 linear extrapolation 而是 phase transition。k=12 进一步加窗，效应是否单调、饱和、还是下降？这关系到 thesis 主线如何论述"actor temporal window 充分性"（claim 1：k=8 足够 vs claim 2：更长窗口持续帮助 vs claim 3：信息饱和后过长窗口反害）。

**本 notebook 任务（pure vanilla + history k=8→12 单变量 ablation）**：把 §7.8 anchor 的 `--history-length 8` 替换成 `--history-length 12`，其它全部不动：

| 维度                  | §7.8 anchor (k=8) | 本 notebook (k=12) |
|---                    |---                |---                 |
| `--history-length`    | 8                 | **12** ← 唯一变量  |
| `--seed`              | 42                | 42                 |
| algorithm             | vanilla SAC       | vanilla SAC        |
| sensor layout         | s0 (DVL-only)     | s0 (DVL-only)      |
| reward                | arrival_v2        | arrival_v2         |
| flow U / target       | 1.5 / 1.5         | 1.5 / 1.5          |
| total_steps           | 1M                | 1M                 |
| num_envs              | 6                 | 6                  |
| benchmark             | `single_u15_cross_tgt15` | 同           |
| obs_dim               | 10×8 + 8 = 88     | **10×12 + 8 = 128**|

**Gate**（与 §7.8 同口径）：
- `final_success_rate ≥ 0.85`
- `last100k_mean ≥ 0.9 × peak`
- `final_oob_rate ≤ 0.10`
- `include_episode_context_obs == True`
- `timeout_bootstrap_semantics == 'terminal'`

**Monotonicity verdict 规则**（驱动 §7.8 续写 + §8 P1#2）：

| Verdict | 触发条件 | 解释 |
|---|---|---|
| **SUPER-MONOTONIC** | k=12 PASS 且 (final ≥ 0.95 OR peak_step earlier by ≥100k OR mean ≥ k=8 mean + 0.05) | k>8 仍持续帮助；建议继续扫 k=16 |
| **MONOTONIC-PLATEAU** | k=12 PASS 且 \|final − 0.900\| ≤ 0.05 且 \|mean − 0.636\| ≤ 0.05 | 饱和点在 k≈8；thesis 主线可定 k=8 |
| **DECAY** | k=12 STRONG-PARTIAL（final ∈ [0.5, 0.85)）| 信息饱和后过长窗口轻微反害（curse of dim / over-stale） |
| **COLLAPSE** | k=12 final < 0.5 | 严重下降；强烈支持 k=8 sweet spot；thesis 写 "actor temporal window 必须 narrow-band tuned" |

**输出根**：
- `experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k12/seed_42/`

**总预算**：~2.5h L4（1 Colab Pro+ session；obs_dim 128 vs 88 增加 ~45% forward cost，但 hidden_dim 仍 256，主要瓶颈是 env step）。

**风格**：训练用 `!python -u -m scripts.train_sac` 直跑，与 §7.8 一致。


## 0. GPU sanity


In [ ]:
!nvidia-smi | head -10


## 1. Mount Drive + cwd


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

REPO_DIR = '/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5'
%cd $REPO_DIR


## 2. Config — single phase（vanilla SAC + history k=12，与 §7.8 anchor 仅差一个 flag value）


In [ ]:
import json
import os
from pathlib import Path

import pandas as pd

# ==== SAC / env config (与 §7.8 anchor 严格一致，除 history_length 外) ====
OBJECTIVE = 'arrival_v2'
PROBE_LAYOUT = 's0'
HISTORY_LENGTH = 12                     # ← 唯一与 §7.8 anchor (k=8) 不同的值
TARGET_SPEED = 1.5
SEED = 42

RANDOM_STEPS = 5_000
UPDATE_AFTER = 5_000
BATCH_SIZE = 256
HIDDEN_DIM = 256
NUM_ENVS = 6
EVAL_EVERY = 25_000
EVAL_EPISODES = 30
CHECKPOINT_EVERY = 100_000
DEVICE = 'cuda'

# 显式拒绝所有 SAC 改进项 — 与 §7.8 一致
USE_ASYMMETRIC_CRITIC = False
USE_LAYERNORM = False
UPDATES_PER_STEP = 1
DROPOUT_RATE = 0.0

PASS_FINAL_SUCCESS = 0.85
PASS_LAST100_RATIO = 0.90
PASS_OOB_RATE = 0.10

# Flow file（与 §7 全套严格一致）
SINGLE_FLOW = 'wake_data/wake_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy'

# ==== Single phase — single_cross s0 k=12 seed=42 (X_*) ====
X_BENCHMARK_KEY = 'single_u15_cross_tgt15'
X_TASK_GEOMETRY = 'cross_stream'
X_FLOW_PATH = SINGLE_FLOW
X_TOTAL_STEPS = 1_000_000
X_RUN_ROOT = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k12/seed_42')
X_CKPT_ROOT = Path('checkpoints/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k12/seed_42')
X_MANIFEST_PATH = Path(f'benchmarks/{X_BENCHMARK_KEY}.json')

# Baselines
X_K8_ANCHOR_ROOT = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k8/seed_42')      # §7.8 anchor
X_K4_S42_ROOT    = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k4/seed_42')      # §7.6.4
X_S1_BASELINE_ROOT = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s1_k4/seed_42')    # §7.1 upper

os.environ['PYTHONUNBUFFERED'] = '1'

print(f'PROBE_LAYOUT          = {PROBE_LAYOUT}')
print(f'OBJECTIVE             = {OBJECTIVE}')
print(f'HISTORY_LENGTH        = {HISTORY_LENGTH}        ← 唯一与 §7.8 anchor (k=8) 不同')
print(f'SEED                  = {SEED}')
print(f'NUM_ENVS              = {NUM_ENVS}')
print(f'USE_ASYMMETRIC_CRITIC = {USE_ASYMMETRIC_CRITIC}')
print(f'USE_LAYERNORM         = {USE_LAYERNORM}')
print(f'UPDATES_PER_STEP      = {UPDATES_PER_STEP}')
print(f'DROPOUT_RATE          = {DROPOUT_RATE}')
print()
print(f'benchmark             : {X_BENCHMARK_KEY}')
print(f'geometry              : {X_TASK_GEOMETRY}')
print(f'total_steps           : {X_TOTAL_STEPS:,}')
print(f'expected obs_dim      : 10 * {HISTORY_LENGTH} + 8 (arrival_v2 context) = {10 * HISTORY_LENGTH + 8}')
print(f'run_root              : {X_RUN_ROOT}')
print(f'§7.8 k=8 anchor       : {X_K8_ANCHOR_ROOT}')
print(f'§7.6.4 k=4 base       : {X_K4_S42_ROOT}')
print(f'§7.1 s1 upper ref     : {X_S1_BASELINE_ROOT}')


## 3. Preflight — flow / arrival_v2 candidate gate / reward unit tests / manifest / baseline 就位


In [ ]:
# Flow file
fp = Path(X_FLOW_PATH)
if not fp.exists():
    raise FileNotFoundError(f'missing flow file: {fp}')
print(f'[OK] flow file: {fp}  ({fp.stat().st_size / 1e6:.1f} MB)')


In [ ]:
!python -u -m scripts.validate_arrival_v2_candidate


In [ ]:
!python -u -m pytest tests/test_reward_objective.py -q


In [ ]:
if not X_MANIFEST_PATH.exists():
    !python -u -m scripts.generate_standard_benchmarks --benchmarks {X_BENCHMARK_KEY} --episodes {EVAL_EPISODES}
if not X_MANIFEST_PATH.exists():
    raise FileNotFoundError(f'manifest not generated: {X_MANIFEST_PATH}')
print(f'[OK] manifest ready: {X_MANIFEST_PATH}')


In [ ]:
# 检查 3 个对比 baseline 是否就位
for label, root, ref_final, ref_oob in [
    ('§7.8 k8 anchor s42  ', X_K8_ANCHOR_ROOT, 0.900, 0.100),
    ('§7.6.4 k4 vanilla s42', X_K4_S42_ROOT, 0.100, 0.667),
    ('§7.1 s1_k4 upper s42 ', X_S1_BASELINE_ROOT, 0.900, 0.100),
]:
    fp = root / 'results' / 'final_eval.json'
    if fp.exists():
        d = json.loads(fp.read_text(encoding='utf-8'))
        f = float(d['eval_success_rate'])
        c = d.get('eval_termination_counts', {})
        n = float(d.get('num_eval_episodes', EVAL_EPISODES))
        oob = float(c.get('out_of_bounds', 0)) / max(n, 1.0)
        match = '✓' if abs(f - ref_final) < 0.05 and abs(oob - ref_oob) < 0.05 else '✗ mismatch'
        print(f'[OK] {label}: final={f:.4f}  oob={oob:.4f}  counts={c}  {match}')
    else:
        print(f'[WARN] {label}: {fp} 不存在 (后续 §5 diff 会回退到 report 转载值)')


## 4. Train — single_cross_s0 + history k=12 + seed=42 (1.0M, skip/resume)


In [ ]:
x_state_path = X_RUN_ROOT / 'trainer_state.json'
if x_state_path.exists():
    x_state = json.loads(x_state_path.read_text(encoding='utf-8'))
    x_current_step = int(x_state.get('env_step', 0))
else:
    x_current_step = 0
print(f'[state] X env_step = {x_current_step:,} / target {X_TOTAL_STEPS:,}')

if x_current_step >= X_TOTAL_STEPS:
    print(f'[skip] X already trained to {x_current_step:,} >= {X_TOTAL_STEPS:,}')
elif x_current_step > 0:
    print(f'[resume] X continuing from {x_current_step:,} -> {X_TOTAL_STEPS:,}')
    !python -u -m scripts.train_sac \
        --resume {str(X_RUN_ROOT)} \
        --total-steps {X_TOTAL_STEPS} \
        --eval-every {EVAL_EVERY} \
        --eval-episodes {EVAL_EPISODES} \
        --checkpoint-every {CHECKPOINT_EVERY} \
        --eval-manifest {str(X_MANIFEST_PATH)} \
        --device {DEVICE}
else:
    print(f'[train] X fresh start -> {X_TOTAL_STEPS:,}')
    !python -u -m scripts.train_sac \
        --flow {X_FLOW_PATH} \
        --task-geometry {X_TASK_GEOMETRY} \
        --target-speed {TARGET_SPEED} \
        --objective {OBJECTIVE} \
        --probe-layout {PROBE_LAYOUT} \
        --history-length {HISTORY_LENGTH} \
        --total-steps {X_TOTAL_STEPS} \
        --random-steps {RANDOM_STEPS} \
        --update-after {UPDATE_AFTER} \
        --batch-size {BATCH_SIZE} \
        --hidden-dim {HIDDEN_DIM} \
        --num-envs {NUM_ENVS} \
        --eval-every {EVAL_EVERY} \
        --eval-episodes {EVAL_EPISODES} \
        --checkpoint-every {CHECKPOINT_EVERY} \
        --eval-manifest {str(X_MANIFEST_PATH)} \
        --seed {SEED} \
        --device {DEVICE} \
        --save-dir {str(X_RUN_ROOT)} \
        --checkpoint-dir {str(X_CKPT_ROOT)}


## 5. Summary + gate


In [ ]:
def summarize_phase(run_root: Path, total_steps: int, label: str, gate_filename: str):
    eval_log_path = run_root / 'results' / 'eval_log.csv'
    final_eval_path = run_root / 'results' / 'final_eval.json'
    trainer_state_path = run_root / 'trainer_state.json'
    train_config_path = run_root / 'results' / 'train_config.txt'

    if not eval_log_path.exists():
        raise FileNotFoundError(f'missing eval log: {eval_log_path}')
    if not final_eval_path.exists():
        raise FileNotFoundError(f'missing final eval: {final_eval_path}')

    df = pd.read_csv(eval_log_path)
    final_eval = json.loads(final_eval_path.read_text(encoding='utf-8'))
    trainer_state = json.loads(trainer_state_path.read_text(encoding='utf-8')) if trainer_state_path.exists() else {}

    history_from_config = 'NA'
    if train_config_path.exists():
        for ln in train_config_path.read_text(encoding='utf-8').splitlines():
            if ln.strip().startswith('history_length='):
                history_from_config = ln.strip().split('=', 1)[1]
                break

    peak_success = float(df['eval_success_rate'].max()) if len(df) else 0.0
    peak_step = int(df.loc[df['eval_success_rate'].idxmax(), 'env_step']) if len(df) else 0
    last100 = df[df['env_step'] >= total_steps - 100_000].copy()
    last100_mean = float(last100['eval_success_rate'].mean()) if len(last100) else 0.0
    mean_all = float(df['eval_success_rate'].mean()) if len(df) else 0.0
    n_evals_with_success = int((df['eval_success_rate'] > 0).sum()) if len(df) else 0
    final_success = float(final_eval['eval_success_rate'])
    counts = final_eval.get('eval_termination_counts', {})
    num_eps = float(final_eval.get('num_eval_episodes', EVAL_EPISODES))
    oob_rate = float(counts.get('out_of_bounds', 0)) / max(num_eps, 1.0)

    print('=' * 100)
    print(f'{label}  (arrival_v2 / s0 / k=12 / seed={SEED} / vanilla / {total_steps:,} steps)')
    print('-' * 100)
    print(f"  final_success_rate    : {final_success:.4f}   gate >= {PASS_FINAL_SUCCESS:.2f}")
    print(f"  peak_success_rate     : {peak_success:.4f}   @ {peak_step:,}")
    print(f"  last100k_mean_success : {last100_mean:.4f}   gate >= {PASS_LAST100_RATIO * peak_success:.4f}")
    print(f"  full-traj mean        : {mean_all:.4f}   (39 evals)")
    print(f"  n_evals_with_success  : {n_evals_with_success} / {len(df)}")
    print(f"  final_oob_rate        : {oob_rate:.4f}   gate <= {PASS_OOB_RATE:.2f}")
    print(f"  obs_dim               : {trainer_state.get('observation_dim', 'NA')}   (expect 10*12+8=128)")
    print(f"  history_length        : {history_from_config}   (from train_config.txt)")
    print(f"  context_obs           : {trainer_state.get('include_episode_context_obs', 'NA')}")
    print(f"  timeout_bootstrap     : {trainer_state.get('timeout_bootstrap_semantics', 'NA')}")
    print(f"  termination           : {counts}")
    print('=' * 100)

    if len(df):
        print()
        print('[last 16 eval rows]')
        cols = ['env_step', 'eval_success_rate', 'eval_return', 'eval_safety_cost',
                'eval_time_s', 'eval_progress_ratio']
        available = [c for c in cols if c in df.columns]
        print(df[available].tail(16).to_string(index=False))

    checks = [
        ('final success >= 0.85', final_success >= PASS_FINAL_SUCCESS, f'{final_success:.4f}'),
        ('last100k mean >= 0.9 * peak', last100_mean >= PASS_LAST100_RATIO * peak_success,
         f'{last100_mean:.4f} / peak={peak_success:.4f}'),
        ('final OOB rate <= 0.10', oob_rate <= PASS_OOB_RATE, f'{oob_rate:.4f}'),
        ('arrival_v2 context obs enabled',
         trainer_state.get('include_episode_context_obs') is True,
         str(trainer_state.get('include_episode_context_obs'))),
        ('arrival_v2 timeout terminal semantics',
         trainer_state.get('timeout_bootstrap_semantics') == 'terminal',
         str(trainer_state.get('timeout_bootstrap_semantics'))),
    ]

    print()
    print('=' * 96)
    print(f"{'check':<48}{'pass':>8}{'detail':>40}")
    print('-' * 96)
    all_pass = True
    for name, ok, detail in checks:
        mark = 'PASS' if ok else 'FAIL'
        if not ok:
            all_pass = False
        print(f'{name:<48}{mark:>8}{detail:>40}')
    print('=' * 96)

    summary = {
        'phase': label,
        'objective': OBJECTIVE,
        'probe_layout': PROBE_LAYOUT,
        'history_length': int(history_from_config) if history_from_config != 'NA' else HISTORY_LENGTH,
        'seed': SEED,
        'total_steps': total_steps,
        'algorithm': 'sac_vanilla',
        'final_success_rate': final_success,
        'peak_success_rate': peak_success,
        'peak_step': peak_step,
        'last100k_mean_success': last100_mean,
        'mean_success_full_trajectory': mean_all,
        'n_evals_with_success': n_evals_with_success,
        'n_evals_total': len(df),
        'final_oob_rate': oob_rate,
        'termination_counts': counts,
        'all_pass': bool(all_pass),
        'checks': [{'name': n, 'ok': bool(ok), 'detail': d} for n, ok, d in checks],
    }
    out_path = run_root / 'results' / gate_filename
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print(f'[saved] {out_path}')
    return summary

x_summary = summarize_phase(
    X_RUN_ROOT,
    X_TOTAL_STEPS,
    'SINGLE_CROSS_S0_K12_SEED42',
    'single_cross_s0_k12_seed42_gate_summary.json',
)
X_PASS = x_summary['all_pass']
print()
print(f'X_PASS = {X_PASS}')


## 6. Monotonicity verdict — k=12 vs k=8 (§7.8 anchor) + k=4 (§7.6.4 floor) + s1_k4 (§7.1 upper)


In [ ]:
def read_baseline(root: Path, ref_final: float, ref_oob: float, ref_mean: float = None):
    fp = root / 'results' / 'final_eval.json'
    log = root / 'results' / 'eval_log.csv'
    out = {'final': ref_final, 'oob': ref_oob, 'mean': ref_mean, 'peak': None, 'peak_step': None, 'source': 'report'}
    if fp.exists():
        d = json.loads(fp.read_text(encoding='utf-8'))
        f = float(d['eval_success_rate'])
        c = d.get('eval_termination_counts', {})
        n = float(d.get('num_eval_episodes', EVAL_EPISODES))
        out['final'] = f
        out['oob'] = float(c.get('out_of_bounds', 0)) / max(n, 1.0)
        out['source'] = 'on-disk'
    if log.exists():
        dlog = pd.read_csv(log)
        out['mean'] = float(dlog['eval_success_rate'].mean())
        out['peak'] = float(dlog['eval_success_rate'].max())
        out['peak_step'] = int(dlog.loc[dlog['eval_success_rate'].idxmax(), 'env_step'])
    return out

print('=' * 100)
print('SINGLE_CROSS — k=8→12 monotonicity verdict (seed=42)')
print('-' * 100)

# 本 run (k=12 seed=42)
k12_final = float(x_summary['final_success_rate'])
k12_oob   = float(x_summary['final_oob_rate'])
k12_peak  = float(x_summary['peak_success_rate'])
k12_peak_step = int(x_summary['peak_step'])
k12_mean  = float(x_summary.get('mean_success_full_trajectory', 0.0))
k12_nsucc = int(x_summary.get('n_evals_with_success', 0))
k12_ntot  = int(x_summary.get('n_evals_total', 0))

# Baselines
b_k8 = read_baseline(X_K8_ANCHOR_ROOT,   0.900, 0.100, 0.636)    # §7.8 anchor
b_k4 = read_baseline(X_K4_S42_ROOT,      0.100, 0.667, 0.221)    # §7.6.4 floor
b_s1 = read_baseline(X_S1_BASELINE_ROOT, 0.900, 0.100, 0.497)    # §7.1 upper

print()
print(f'{"config":<48}{"final":>10}{"mean":>10}{"oob":>10}{"peak":>10}{"peak@":>14}')
print('-' * 100)
print(f'{"k=4 vanilla s42 (§7.6.4 FAIL floor)":<48}{b_k4["final"]:>10.4f}{(b_k4["mean"] or float("nan")):>10.4f}{b_k4["oob"]:>10.4f}{(b_k4["peak"] or float("nan")):>10.4f}{(b_k4["peak_step"] or 0):>14,}')
print(f'{"k=8 vanilla s42 (§7.8 anchor PASS)":<48}{b_k8["final"]:>10.4f}{(b_k8["mean"] or float("nan")):>10.4f}{b_k8["oob"]:>10.4f}{(b_k8["peak"] or float("nan")):>10.4f}{(b_k8["peak_step"] or 0):>14,}')
print(f'{"k=12 vanilla s42 (THIS RUN)":<48}{k12_final:>10.4f}{k12_mean:>10.4f}{k12_oob:>10.4f}{k12_peak:>10.4f}{k12_peak_step:>14,}')
print(f'{"s1_k4 vanilla s42 (§7.1 upper ref)":<48}{b_s1["final"]:>10.4f}{(b_s1["mean"] or float("nan")):>10.4f}{b_s1["oob"]:>10.4f}{(b_s1["peak"] or float("nan")):>10.4f}{(b_s1["peak_step"] or 0):>14,}')
print('=' * 100)

# Δ 行
delta_final_vs_k8 = k12_final - b_k8['final']
delta_mean_vs_k8  = k12_mean  - (b_k8['mean'] or 0.0)
delta_oob_vs_k8   = k12_oob   - b_k8['oob']
delta_peak_step   = k12_peak_step - (b_k8['peak_step'] or 0)

print()
print(f'{"contrast":<60}{"Δ final":>12}{"Δ mean":>12}{"Δ oob":>12}{"Δ peak@":>14}')
print('-' * 110)
print(f'{"k=12 vs k=8 (§7.8 anchor) — monotonicity test":<60}'
      f'{delta_final_vs_k8:>+12.4f}{delta_mean_vs_k8:>+12.4f}{delta_oob_vs_k8:>+12.4f}{delta_peak_step:>+14,}')
print(f'{"k=12 vs k=4 (§7.6.4 floor) — cumulative gain":<60}'
      f'{k12_final - b_k4["final"]:>+12.4f}'
      f'{(k12_mean - (b_k4["mean"] or 0)):>+12.4f}'
      f'{k12_oob - b_k4["oob"]:>+12.4f}'
      f'{"N/A":>14}')
print(f'{"k=12 vs s1_k4 (§7.1 upper) — gap-to-upper":<60}'
      f'{k12_final - b_s1["final"]:>+12.4f}'
      f'{(k12_mean - (b_s1["mean"] or 0)):>+12.4f}'
      f'{k12_oob - b_s1["oob"]:>+12.4f}'
      f'{"N/A":>14}')
print('=' * 110)
print()
print(f'k=12 evals_with_success: {k12_nsucc} / {k12_ntot}  (k=8 was 37/39, k=4 was 35/39)')

# Monotonicity verdict — 4 档
k12_pass = (k12_final >= PASS_FINAL_SUCCESS) and (k12_oob <= PASS_OOB_RATE)
k12_strong_partial = (k12_final >= 0.50) and (k12_final < PASS_FINAL_SUCCESS)
k12_collapse = (k12_final < 0.50)

super_monotonic = (
    k12_pass and (
        k12_final >= 0.95 or
        delta_peak_step <= -100_000 or
        delta_mean_vs_k8 >= 0.05
    )
)
plateau = (
    k12_pass and
    abs(delta_final_vs_k8) <= 0.05 and
    abs(delta_mean_vs_k8) <= 0.05 and
    not super_monotonic
)

if super_monotonic:
    verdict = (
        'SUPER-MONOTONIC — k=12 比 k=8 还好（更高 final / 更早 peak / 更高 mean）；'
        'temporal window 未饱和；建议 §8 P1#2 续扫 k=16，看是否还能进一步压低 peak_step'
    )
elif plateau:
    verdict = (
        'MONOTONIC-PLATEAU — k=12 ≈ k=8（|Δfinal|≤0.05 且 |Δmean|≤0.05）；'
        '饱和点在 k≈8；thesis 主线可定 k=8 为 sweet spot，无需 k=16；'
        '§7.8 续写 "monotonic saturation observed at k=8"'
    )
elif k12_pass and not plateau:
    verdict = (
        'PASS-WITH-SHIFT — k=12 仍 PASS 但 Δfinal 或 Δmean 偏离 plateau 阈值；'
        '可能是 mild-DECAY（mean 略降）或 SUB-OPTIMAL（peak_step 推迟）；'
        '建议 §8 P1#2 续扫 k=16 + multi-seed 才能定论'
    )
elif k12_strong_partial:
    verdict = (
        'DECAY — k=12 STRONG-PARTIAL（final ∈ [0.5, 0.85)）；'
        '信息饱和后过长窗口轻微反害（curse of dim / over-stale history）；'
        '强证据 k=8 是 sweet spot；thesis 写 "actor temporal window 必须 narrow-band tuned"'
    )
else:  # k12_collapse
    verdict = (
        'COLLAPSE — k=12 final<0.50；严重下降；'
        '强烈支持 k=8 sweet spot；thesis 写 "k>8 的 actor window 导致 information dilution / network capacity overload"；'
        '同时启动 audit（确认非训练 bug：obs_dim、grad-norm、loss curve）'
    )

print()
print(f'>>> verdict: {verdict}')

# 落盘 monotonicity summary
mono_out = {
    'experiment': 'arrival_v2_s0_cross_k12_seed42_monotonicity',
    'seed': SEED,
    'benchmark': X_BENCHMARK_KEY,
    'probe_layout': PROBE_LAYOUT,
    'history_length': HISTORY_LENGTH,
    'total_steps': X_TOTAL_STEPS,
    'algorithm': 'sac_vanilla',
    'cli_diff_vs_§7.8_anchor': '--history-length 8 → 12',
    'results': {
        'k12_s42_thisrun': {
            'final': k12_final, 'mean': k12_mean, 'oob': k12_oob,
            'peak': k12_peak, 'peak_step': k12_peak_step,
            'n_evals_with_success': k12_nsucc, 'n_evals_total': k12_ntot,
        },
        'k8_s42_anchor_§7.8': b_k8,
        'k4_s42_floor_§7.6.4': b_k4,
        's1_k4_s42_upper_§7.1': b_s1,
    },
    'delta_vs_§7.8_k8_anchor': {
        'final_pp': round(delta_final_vs_k8 * 100, 2),
        'mean_pp':  round(delta_mean_vs_k8  * 100, 2),
        'oob_pp':   round(delta_oob_vs_k8   * 100, 2),
        'peak_step_shift': int(delta_peak_step),
    },
    'all_pass': bool(x_summary['all_pass']),
    'verdict': verdict,
}
out_dir = Path('experiments/arrival_v2_prototype/s0_cross_k12_seed42_summary')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'combined_gate_summary.json'
out_path.write_text(json.dumps(mono_out, indent=2), encoding='utf-8')
print(f'\n[saved] {out_path}')
